# Custom Non-LLM Feedback Functions

TruLens feedback functions don't have to call an LLM. Many useful evaluations are
purely programmatic: regex checks, schema validation, length constraints, and PII
detection. These run instantly, cost nothing, and are fully deterministic.

This notebook shows four non-LLM feedback functions:

1. **JSON schema check** — verify the response is valid JSON matching a schema
2. **Regex match** — check that the response matches an expected pattern
3. **Response length** — score based on response length being within a target range
4. **PII regex scan** — detect personally identifiable information in the response

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/truera/trulens/blob/main/examples/cookbooks/custom_programmatic_feedback.ipynb)

In [ ]:
# !pip install trulens openai

## Setup

In [ ]:
import json
import os
import re

from openai import OpenAI
from trulens.core import TruSession
from trulens.apps.custom import TruCustomApp
from trulens.core.feedback import Feedback
from trulens.core.otel.instrument import instrument
from trulens.otel.semconv.trace import SpanAttributes

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = "sk-proj-..."

session = TruSession()
session.reset_database()

## Define a simple app

We build a minimal app that returns structured JSON responses — a good candidate
for programmatic evaluation.

In [ ]:
client = OpenAI()


class StructuredResponseApp:
    """App that returns JSON-structured responses."""

    SYSTEM_PROMPT = """You are a helpful assistant. Always respond with valid JSON.
Your response must be a JSON object with exactly these fields:
- "answer": string (your answer to the question)
- "confidence": number between 0 and 1
- "sources": list of strings (data sources you'd reference)
"""

    @instrument(
        span_type=SpanAttributes.SpanType.RECORD_ROOT,
        attributes={
            SpanAttributes.RECORD_ROOT.INPUT: "query",
            SpanAttributes.RECORD_ROOT.OUTPUT: "return",
        },
    )
    def respond(self, query: str) -> str:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": self.SYSTEM_PROMPT},
                {"role": "user", "content": query},
            ],
        )
        return response.choices[0].message.content or ""


app = StructuredResponseApp()

## Feedback function 1: JSON schema check

Returns 1.0 if the response is valid JSON with the required fields and correct
value types. Returns 0.0 otherwise.

In [ ]:
REQUIRED_SCHEMA = {
    "answer": str,
    "confidence": (int, float),
    "sources": list,
}


def json_schema_check(response: str) -> float:
    """Check that the response is valid JSON matching the required schema.

    Args:
        response: The app's response string.

    Returns:
        1.0 if response is valid JSON with all required fields and correct types,
        0.0 otherwise.
    """
    try:
        data = json.loads(response)
    except (json.JSONDecodeError, TypeError):
        return 0.0

    if not isinstance(data, dict):
        return 0.0

    for field, expected_type in REQUIRED_SCHEMA.items():
        if field not in data:
            return 0.0
        if not isinstance(data[field], expected_type):
            return 0.0

    # confidence must be in [0, 1]
    if not 0.0 <= data["confidence"] <= 1.0:
        return 0.0

    return 1.0


# Quick sanity checks
assert json_schema_check('{"answer": "yes", "confidence": 0.9, "sources": ["wiki"]}') == 1.0
assert json_schema_check('{"answer": "yes"}') == 0.0  # missing fields
assert json_schema_check("not json") == 0.0
print("json_schema_check: OK")

## Feedback function 2: Regex match

Checks that the response (or a parsed field) matches a specific pattern.
Useful for format compliance — e.g., dates, codes, citation styles.

In [ ]:
def answer_not_empty(response: str) -> float:
    """Check that the 'answer' field is non-empty (at least 10 characters).

    Args:
        response: The app's response string.

    Returns:
        1.0 if the answer field has >= 10 characters, 0.0 otherwise.
    """
    try:
        data = json.loads(response)
        answer = data.get("answer", "")
    except (json.JSONDecodeError, TypeError, AttributeError):
        # Fall back to treating the whole response as the answer
        answer = response

    pattern = re.compile(r".{10,}", re.DOTALL)
    return 1.0 if pattern.search(str(answer)) else 0.0


assert answer_not_empty('{"answer": "This is a complete answer.", "confidence": 0.8, "sources": []}') == 1.0
assert answer_not_empty('{"answer": "short", "confidence": 0.8, "sources": []}') == 0.0
print("answer_not_empty: OK")

## Feedback function 3: Response length score

Scores the response based on how close its length is to a target range.
Returns 1.0 if within range, scales down linearly as the response drifts
outside the range.

In [ ]:
TARGET_MIN_CHARS = 50
TARGET_MAX_CHARS = 500


def response_length_score(response: str) -> float:
    """Score the response based on character length being within a target range.

    Returns 1.0 if within [TARGET_MIN_CHARS, TARGET_MAX_CHARS],
    scales linearly to 0.0 as the response drifts outside the range.

    Args:
        response: The app's response string.

    Returns:
        Float in [0.0, 1.0].
    """
    length = len(response)

    if TARGET_MIN_CHARS <= length <= TARGET_MAX_CHARS:
        return 1.0

    if length < TARGET_MIN_CHARS:
        # Too short: score = length / min
        return max(0.0, length / TARGET_MIN_CHARS)

    # Too long: score decreases as length grows past max
    overage = length - TARGET_MAX_CHARS
    tolerance = TARGET_MAX_CHARS  # allow up to 2x max before hitting 0
    return max(0.0, 1.0 - overage / tolerance)


assert response_length_score("x" * 100) == 1.0   # within range
assert response_length_score("x" * 25) == 0.5    # too short
assert response_length_score("x" * 1000) == 0.0  # way too long
print("response_length_score: OK")

## Feedback function 4: PII detection

Scans the response for common PII patterns: email addresses, phone numbers,
and US Social Security Numbers. Returns 1.0 if no PII detected (clean),
0.0 if PII is found.

In [ ]:
PII_PATTERNS = [
    # Email addresses
    re.compile(r"[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}"),
    # US phone numbers (various formats)
    re.compile(r"\b(?:\+?1[\s\-.]?)?\(?\d{3}\)?[\s\-.]\d{3}[\s\-.]\d{4}\b"),
    # US Social Security Numbers
    re.compile(r"\b\d{3}[\-\s]\d{2}[\-\s]\d{4}\b"),
]


def no_pii(response: str) -> float:
    """Check that the response contains no common PII patterns.

    Scans for email addresses, phone numbers, and Social Security Numbers.

    Args:
        response: The app's response string.

    Returns:
        1.0 if no PII detected, 0.0 if PII is found.
    """
    for pattern in PII_PATTERNS:
        if pattern.search(response):
            return 0.0
    return 1.0


assert no_pii("The answer is 42.") == 1.0
assert no_pii("Contact me at user@example.com") == 0.0
assert no_pii("Call 555-123-4567") == 0.0
assert no_pii("SSN: 123-45-6789") == 0.0
print("no_pii: OK")

## Wire feedback functions to TruLens

Wrap each function in a `Feedback` object. Non-LLM feedback functions receive
span attributes directly via selectors — here we read the record output.

In [ ]:
from trulens.core.schema.select import Select

f_json_schema = (
    Feedback(json_schema_check, name="JSON Schema Check")
    .on(Select.RecordOutput)
)

f_answer_not_empty = (
    Feedback(answer_not_empty, name="Answer Not Empty")
    .on(Select.RecordOutput)
)

f_length = (
    Feedback(response_length_score, name="Response Length")
    .on(Select.RecordOutput)
)

f_no_pii = (
    Feedback(no_pii, name="No PII")
    .on(Select.RecordOutput)
)

feedbacks = [f_json_schema, f_answer_not_empty, f_length, f_no_pii]

## Record and evaluate

In [ ]:
tru_app = TruCustomApp(
    app,
    app_name="StructuredResponseApp",
    app_version="v1",
    feedbacks=feedbacks,
)

test_queries = [
    "What is the capital of France?",
    "How does photosynthesis work?",
    "What are the main causes of climate change?",
    "Explain the difference between RAM and ROM.",
]

with tru_app as recording:
    for query in test_queries:
        response = app.respond(query)
        print(f"Q: {query}")
        print(f"A: {response[:80]}...\n")

## View results

In [ ]:
session.get_leaderboard()

In [ ]:
records, feedback_results = session.get_records_and_feedback()
records[["input", "output", "JSON Schema Check", "Answer Not Empty", "Response Length", "No PII"]]

## Launch the dashboard

Explore per-record scores in the TruLens dashboard.

In [ ]:
from trulens.dashboard import run_dashboard

run_dashboard(session)

## Summary

Non-LLM feedback functions are:

- **Free** — no LLM calls, no API cost
- **Instant** — run in microseconds
- **Deterministic** — same input always produces the same score
- **Composable** — mix with LLM-based feedback functions in the same `TruApp`

Use them for format compliance, safety checks, and any evaluation where
correctness can be determined without a language model.